前言：第五篇，给 Agent 发一张"工单"
上一篇，咱们给循环装上了挂钩：日志、权限、自动收尾、会话统计，全挂在外面，循环本体一行没动。加功能从"开胸手术"变成了"往墙上挂东西"。到这，这个 Agent 看起来已经挺像样了：会干活（五个工具）、有分寸（门禁）、可扩展（挂钩）。

但有个问题，从第一篇憋到现在，我一直没敢碰：这货干活，没有计划。

给它一个多步任务，它一上来就闷头写。写两步，跑个测试，蹦出个报错，它去修报错，修着修着——它忘了自己最开始是要干嘛的了。你盯着终端能明显感觉到那个瞬间：刚才还目标明确的一个人，忽然开始即兴发挥。

工具没问题，权限没问题，挂钩也没问题。问题出在一个更隐蔽的地方：执行到一半的时候，目标不在它的视野里了。

这一篇就干一件事：给 Agent 发一张工单——一个叫 todo_write 的工具。动手之前先列清单，做一项勾一项，清单本身回流进上下文，让"我在哪、我该干嘛"变成它随时看得见的东西。

读完你会拿到三样东西：

TodoManager
 的完整实现：一块带状态机的白板，外加三条专治"清单走样"的校验规则
一个反直觉的设计洞察：这个工具不给 Agent 增加任何执行能力——它连一个字节都不读写，它增加的是规划状态
本篇的高潮：reminder 机制——模型列完清单就扔在一边不更新？harness 会主动戳它一下；以及一个"连戳带勾"的十步任务实录
门槛不变：会 Python 基础语法、有上一篇跑通的那份带挂钩的代码。直接开始。

PART 01：案发现场——做到第四步，它忘了自己是来干嘛的
先复原现场。我给上一版的 Agent 出过一道题，这道题出自源仓库的教程，堪称"注意力测试"的经典款：

把所有 Python 文件改成 snake_case 命名，然后跑测试，修好失败的。

任务本身不难，拆开就三步：重命名、跑测试、修失败。任何一个实习生拿到这张工单，都会先在纸上写下这三行。

但模型的实际表现是这样的。开局其实很正常，甚至可以说漂亮：

> glob("**/*.py")
> bash("git mv HelloWorld.py hello_world.py")
> bash("git mv UserData.py user_data.py")
> bash("python -m pytest")
有条有理：找文件、改命名、跑测试。然后测试蹦出 2 个失败，画风开始变了：

> bash("python -m pytest tests/test_user_data.py")
> read_file("tests/test_user_data.py")
> edit_file("src/user_data.py", ...)
> edit_file("src/user_data.py", ...)   # 修第 1 个失败，顺手重构了一段
> bash("python -m pytest")             # 重构引入了新问题
> edit_file("src/helpers.py", ...)     # 去处理新问题
看到问题了吗？从第 5 行开始，它在响应报错，而不是在执行任务。每一条新命令的动机都是上一条命令的输出，像多米诺骨牌一样一张推一张——而"把所有文件改成 snake_case"这个最初的目标，不在任何一张牌上。

最后它跟我说"任务完成"的时候，项目里还躺着 4 个没改名的文件。它不装傻，也不偷懒，它就是纯粹地、诚恳地——忘了。因为它此刻的注意力里，确实没有什么没完成的事。

后面的文件没有重命名。它不装傻，也不偷懒，它就是纯粹地、诚恳地——忘了。最后它跟我说"任务完成"的时候，语气特别坦然，因为它此刻的注意力里，确实没有什么没完成的事。

注意，这个跑偏过程里没有任何一步是错的。跑测试是对的，修失败也是对的——单看每一轮，它的决策都合理。灾难是连续性造成的：一步合理的响应接着一步合理的响应，走着走着就走到了任务外面。

为什么会这样？得从这个"脑子"的物理结构说起。

模型的注意力，天然偏向最近的内容。对话每多一轮，工具结果就往上下文里灌一大块新东西——文件内容、命令输出、报错堆栈，每一样都又长又吵。而任务目标呢，是开头那一小段话，离最新的 token 越来越远，影响力被一步步稀释。

一个十步的任务，做完 1-3 步之后，4-10 步就已经被挤出注意力中心了。它不是态度差，是记忆的物理结构就这样：每次只盯着最近的东西看。

这事人也一样。让你边接电话边找文件边回消息，十分钟后你也会忘了最初要干嘛——所以人才发明了便利贴、待办清单、工单系统。你的记忆没变，只是目标被写到了一个你随时看得见的地方。

给 Agent 也发一张，就是这一篇全部的活。

PART 02：TodoManager——一块会自己画勾的白板
先看数据结构。整个方案的核心就一个类，持有内存里的一张清单：

In [ ]:
from typing import List, Dict, Union

class TodoManager:
    def __init__(self):
        # 存储todo列表，每一项是字典: {"content": 任务文本, "status": 状态}
        self.items: list[dict] = []

    def update(self, todos: Union[List[Dict], str]) -> str:
        """
        更新todo列表，做数据校验，替换当前items，最后返回渲染后的文本
        todos: 支持两种输入
            1. list[dict]：todo对象列表
            2. str：可以是多行文本，用来解析成todo（可选扩展）
        """
        validated = []
        allowed_status = {"pending", "in_progress", "completed"}

        # ===== 校验逻辑 =====
        if isinstance(todos, list):
            for item in todos:
                # 必须包含content和status字段
                if not isinstance(item, dict):
                    continue
                if "content" not in item or "status" not in item:
                    continue
                # status只能是规定的三种
                status = item["status"]
                if status not in allowed_status:
                    continue
                # content转字符串，简单清洗
                content = str(item["content"]).strip()
                if not content:
                    continue
                validated.append({"content": content, "status": status})
        elif isinstance(todos, str):
            # 可选：如果传入字符串，你可以在这里写文本解析todo的逻辑
            # 例如解析 "[ ] 买牛奶" 这种行
            pass

        # 替换todo列表
        self.items = validated
        # 调用render返回格式化字符串
        return self.render()

    def render(self) -> str:
        """把todo列表渲染成人类可读的文本"""
        if not self.items:
            return "No todos."

        lines = []
        for todo in self.items:
            marker = {
                "pending": "[ ]",
                "in_progress": "[>]",
                "completed": "[x]",
            }[todo["status"]]
            lines.append(f"{marker} {todo['content']}")

        # 统计完成数量
        done = sum(t["status"] == "completed" for t in self.items)
        lines.append(f"\n({done}/{len(self.items)} completed)")
        return "\n".join(lines)


# 全局实例
TODO = TodoManager()

每项就两个键：content（干什么）和 status（什么状态）。状态是个三档状态机：

[ ]
 pending——还没动工
[>]
 in_progress——正在干
[x]
 completed——干完了
渲染出来长这样，一张人话写的进度表：

[ ] Rename hello.py to snake_case
[>] Run tests
[x] Read existing files
(1/3 completed)
末尾那个 (1/3 completed) 是点睛之笔——进度条。模型每次更新清单，都会看见自己走到哪了。

但真正值钱的是 update() 里的三条校验。每一条都在治一种真实的"清单病"：

第一条：最多 20 项。

iflen(todos) >20:
raiseValueError("Max 20 todos allowed")
清单太长等于没有清单。20 项排开，注意力又被稀释回 PART 01 的老问题。撞到上限，说明这个任务本身太大了，该拆任务，而不是硬塞——这个边界我们在下一篇会真正用上，先埋个伏笔。

第二条：content 必须非空。

content =str(todo.get("content", "")).strip()
ifnot content:
raiseValueError(f"todos[{index}] requires content")
空项是自欺欺人的清单。"待办：干点啥"这种条目，列了等于没列，还占用了注意力。要么写清楚，要么别写。

第三条，也是我觉得最妙的一条：同一时间，只允许一个 in_progress。

if in_progress >1:
raiseValueError("Only one todo can be in_progress at a time")
这条在治什么？治"全都开着头，全都没收尾"。模型特别容易犯这个病：干着 A 想起 B，把 B 也标成进行中，干着 B 又觉得 C 急，C 也开着——最后一张清单上五六个 [>]，跟没标一样。强制单线程推进，每一时刻"正在干什么"有且只有一个答案。

还有一个细节值得单独说：update() 的入参类型是 list | str。模型调工具时传参数，可能规规矩矩传 JSON 数组，也可能手滑传个 JSON 字符串，甚至传个 Python 列表的字符串表示。代码的处理是先 json.loads，失败再 ast.literal_eval：

In [ ]:
import json
import ast

if isinstance(todos, str):
    try:
        todos = json.loads(todos)
    except json.JSONDecodeError:
        try:
            todos = ast.literal_eval(todos)
        except (SyntaxError, ValueError) as e:
            raise ValueError("todos must be a list or JSON array string") from e

注意用的是 ast.literal_eval，坚决不用 eval——后者会执行任意代码，模型传进来一句 "__import__('os').system('rm -rf /')"，你的机器就没了。上一篇装好的门禁管的是工具调用，管不到你在 handler 里手贱写的 eval。安全这根弦，每一行都得绷着。

最后，最关键的一步藏在返回值里：update() 返回的是 render() 之后的清单文本。这个文本会作为 tool_result 回流进上下文——

模型每勾一项，就重新看见一遍整张清单。这就是"看得见"的全部秘密，没有魔法。